# Trading strategy

The game in one line: **settlement = sum of `N` draws `X_i ~ Uniform(a, b)`**, where `(a, w=b-a)` are drawn once at the start of each round from two LogNormal-floored priors and never change for that round. We see one `X_i` every `reveal_interval` seconds.

The right thing to trade is the conditional expectation of the final sum given the reveals we've seen so far. Everything below is built around computing that posterior cleanly and trading the spread between our fair value and the book.

### Pieces

1. **`Posterior`** — a discrete joint posterior over `(a, w)` with the proper prior from the handout. Updates in closed form on each reveal using the Uniform likelihood and its support constraint. Exposes `predict_settle(running_sum, n_remaining) -> (mean, std)`.

2. **`Strategy`** — owns the posterior, our position, and our two resting orders. Every event:
   - reconciles position against the server,
   - recomputes fair value `F` and standard deviation `sigma`,
   - **snipes** any quote on the book whose mispricing vs `F` exceeds `taker_fee + buffer * sigma`,
   - posts a passive bid at `F - edge` and ask at `F + edge`, with `edge` scaling with `sigma` and skewed by inventory,
   - uses `client.modify()` to reprice without losing the order if possible.

3. **Callbacks** wire reveals/fills/phase-changes into the strategy. The WS feed drives everything; no polling.

### What is *not* done here (intentionally)

- No assumption that the market is efficient. We trust our fair, not the book.
- No bot-pattern fitting. The handout warns parameters may change — we only encode the *distributional form*, with parameters tunable at the top of the posterior cell.
- No per-round parameter tuning. Knobs at the top of `Strategy.__init__` are the only tunables, and they're orthogonal (spread, inventory aversion, snipe aggressiveness).

## 1. Connect

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import math
import random
import threading
from collections import defaultdict

from sdk.client import GameClient

URL = "http://192.168.50.167:8000"
API_KEY = "intern2-KEVD"

c = GameClient(URL, API_KEY)
c.game_state()

## 2. Posterior over `(a, w)`

We construct the prior over `(a, w)` by Monte Carlo from the LogNormal-floored definitions in the handout, then turn it into a discrete probability dictionary keyed by `(a, w)`.

Each reveal `x` updates the posterior via Bayes:

$$P(a, w \mid x_{1:k}) \propto P(a, w) \prod_{i=1}^{k} \mathbf{1}[a \le x_i \le a+w] \cdot \frac{1}{w}$$

We then derive:

- `mean_x()` &mdash; current `E[X | data]`.
- `predict_settle(running_sum, n_remaining)` &mdash; mean and std of the **final settlement**, using the law of total variance to combine within-round draw variance and posterior uncertainty over `(a, w)`.

The prior parameters live at the top of the next cell. If the admin announces a parameter change later in the week, this is the only place to edit.

In [ ]:
# Distributional parameters from the handout. Tweak if announced.
A_LOGN_MU, A_LOGN_SIGMA = 1.0, 0.8   # a  = floor(LogN(mu, sigma))
W_LOGN_MU, W_LOGN_SIGMA = 0.5, 0.7   # w  = 1 + floor(LogN(mu, sigma))
N_PRIOR_SIM = 1_000_000              # MC samples to build the discrete prior


class Posterior:
    """Discrete joint posterior over (a, w) given reveals X_i ~ Uniform(a, a+w)."""

    def __init__(
        self,
        a_mu: float = A_LOGN_MU,
        a_sigma: float = A_LOGN_SIGMA,
        w_mu: float = W_LOGN_MU,
        w_sigma: float = W_LOGN_SIGMA,
        n_sim: int = N_PRIOR_SIM,
    ):
        counts: dict[tuple[int, int], int] = defaultdict(int)
        for _ in range(n_sim):
            a_v = int(math.floor(random.lognormvariate(a_mu, a_sigma)))
            w_v = 1 + int(math.floor(random.lognormvariate(w_mu, w_sigma)))
            counts[(a_v, w_v)] += 1
        total = sum(counts.values())
        self.prior: dict[tuple[int, int], float] = {k: v / total for k, v in counts.items()}
        self.posterior: dict[tuple[int, int], float] = dict(self.prior)
        self.reveals: list[float] = []

    def reset(self, reveals: list[float] | None = None) -> None:
        self.posterior = dict(self.prior)
        self.reveals = []
        for x in reveals or []:
            self.update(x)

    def update(self, x: float) -> None:
        self.reveals.append(float(x))
        new: dict[tuple[int, int], float] = {}
        total = 0.0
        eps = 1e-9
        for (a, w), p in self.posterior.items():
            b = a + w
            if x < a - eps or x > b + eps:
                continue
            new_p = p / w  # uniform likelihood 1/(b-a) = 1/w
            new[(a, w)] = new_p
            total += new_p
        if total <= 0:
            # Observation outside support of every (a,w) in our discrete grid.
            # Don't poison the posterior; warn so we can investigate.
            print(f"WARN: reveal {x} outside posterior support; ignoring update")
            return
        self.posterior = {k: v / total for k, v in new.items()}

    def mean_x(self) -> float:
        return sum(p * (a + w / 2.0) for (a, w), p in self.posterior.items())

    def predict_settle(self, running_sum: float, n_remaining: int) -> tuple[float, float]:
        """Mean and standard deviation of the total final settlement.

        Uses the law of total variance:
            Var[S_rem] = E[ Var[S_rem | a,w] ]  +  Var[ E[S_rem | a,w] ]
        where S_rem is the sum of the n_remaining unseen draws.
        """
        if n_remaining <= 0:
            return float(running_sum), 0.0
        e_inner = 0.0   # E[ E[S_rem | a,w] ]
        e2_inner = 0.0  # E[ E[S_rem | a,w]^2 ]
        e_var = 0.0     # E[ Var[S_rem | a,w] ]
        for (a, w), p in self.posterior.items():
            inner_mean = n_remaining * (a + w / 2.0)
            inner_var = n_remaining * (w * w) / 12.0
            e_inner += p * inner_mean
            e2_inner += p * inner_mean * inner_mean
            e_var += p * inner_var
        var_total = e_var + (e2_inner - e_inner * e_inner)
        return running_sum + e_inner, math.sqrt(max(var_total, 0.0))


post = Posterior()
print(f"Prior support has {len(post.prior)} distinct (a, w) pairs.")
mean0, std0 = post.predict_settle(running_sum=0, n_remaining=10)
print(f"Prior settle estimate with N=10, k=0: {mean0:.1f} +/- {std0:.1f}")

## 3. Strategy

`Strategy` owns the posterior, our position, and (at most) one resting bid and one resting ask. Every event ends up in `step()`, which:

1. **Reconciles** our local position against `c.positions()` (cheap insurance against any local drift).
2. Computes desired quotes from the posterior:
   - `edge = max(min_edge, edge_per_sigma * sigma)` — half-spread widens with uncertainty.
   - `skew = -position * skew_per_unit` — long inventory pushes both bid and ask **down** so we lean toward selling, and vice versa.
3. **Snipes** if the book is mispriced: takes via IOC when `expected_profit_per_unit > taker_fee + snipe_buffer * sigma`. Sizes the take to whatever's on the book up to `quote_qty` and the remaining position-limit headroom.
4. **Quotes** the two passive sides. Uses `client.modify()` to reprice if there's already a resting order; falls back to cancel+repost only if modify fails.
5. Refuses to post a side that would push position beyond `position_limit - quote_qty`.

All state mutations and order placements are protected by a reentrant lock so the WS thread and the main thread can both call `step()` safely.

In [ ]:
class Strategy:
    def __init__(self, client: GameClient, posterior: Posterior, symbol: str = "A"):
        self.c = client
        self.symbol = symbol
        self.posterior = posterior
        self.lock = threading.RLock()

        gs = client.game_state()
        self.duration = gs["duration"]
        self.reveal_interval = gs["reveal_interval"]
        self.n_total = self.duration // self.reveal_interval
        instr = gs["instruments"][symbol]
        self.tick = instr["tick_size"]
        self.position_limit = instr["position_limit"]
        fees = gs.get("fees", {})
        self.maker_fee = fees.get("maker_per_lot", 0.5)
        self.taker_fee = fees.get("taker_per_lot", 0.5)

        # Seed posterior with whatever reveals already happened in the current round.
        self.posterior.reset(gs.get("reveals") or [])

        # Position from server, not a local guess.
        self.position = int(client.positions()["positions"].get(symbol, 0))

        # resting[side] -> {"order_id", "price", "qty"} or None
        self.resting: dict[str, dict | None] = {"bid": None, "ask": None}

        # ---------- knobs ----------
        self.quote_qty = 10           # size per side
        self.min_edge = 1.5           # half-spread floor (must exceed maker_fee + taker_fee/2 for round-trip + slack)
        self.edge_per_sigma = 0.25    # half-spread per unit of settle-std
        self.skew_per_unit = 0.10     # bid/ask shift per unit of inventory
        self.snipe_buffer_sigma = 0.30  # how many sigmas of fair-value uncertainty we require above taker_fee to snipe
        self.snipe_min_edge = 0.5     # absolute minimum profit-per-unit above taker_fee
        # ----------------------------

    # -------- helpers --------

    def _running_sum(self) -> float:
        return sum(self.posterior.reveals)

    def _n_remaining(self) -> int:
        return max(self.n_total - len(self.posterior.reveals), 0)

    def fair_and_sigma(self) -> tuple[float, float]:
        return self.posterior.predict_settle(self._running_sum(), self._n_remaining())

    def reconcile_position(self) -> None:
        try:
            self.position = int(self.c.positions()["positions"].get(self.symbol, 0))
        except Exception as e:
            print(f"reconcile_position failed: {e}")

    def desired_quotes(self) -> tuple[int | None, int | None, float, float]:
        fair, sigma = self.fair_and_sigma()
        edge = max(self.min_edge, self.edge_per_sigma * sigma)
        skew = -self.position * self.skew_per_unit

        bid_px = int(math.floor(fair - edge + skew))
        ask_px = int(math.ceil(fair + edge + skew))
        if ask_px <= bid_px:
            ask_px = bid_px + self.tick

        # Respect position limit with quote_qty headroom.
        if self.position + self.quote_qty > self.position_limit:
            bid_px = None
        if self.position - self.quote_qty < -self.position_limit:
            ask_px = None

        return bid_px, ask_px, fair, sigma

    # -------- order plumbing --------

    def _record_fill_from_trades(self, side: str, trades: list[dict]) -> int:
        """Apply fills returned synchronously from a REST call. Returns total filled."""
        filled = sum(t["qty"] for t in trades)
        if filled:
            self.position += filled if side == "buy" else -filled
        return filled

    def _safe_cancel(self, side: str) -> None:
        rest = self.resting[side]
        if rest is None:
            return
        try:
            self.c.cancel(rest["order_id"])
        except Exception:
            pass
        self.resting[side] = None

    def _post(self, side: str, price: int) -> None:
        method = self.c.buy if side == "bid" else self.c.sell
        sgn_side = "buy" if side == "bid" else "sell"
        try:
            r = method(self.symbol, price=price, qty=self.quote_qty)
        except Exception as e:
            self.resting[side] = None
            print(f"  post {side} @ {price} failed: {e}")
            return
        o = r["order"]
        self._record_fill_from_trades(sgn_side, r.get("trades", []))
        if o["status"] in ("open", "partial"):
            self.resting[side] = {
                "order_id": o["order_id"],
                "price": o["price"],
                "qty": o["remaining"],
            }
        else:
            self.resting[side] = None

    def _reprice(self, side: str, want_px: int) -> None:
        rest = self.resting[side]
        if rest is None:
            self._post(side, want_px)
            return
        if rest["price"] == want_px and rest["qty"] == self.quote_qty:
            return  # already where we want, full size — leave it alone (keep queue priority)
        sgn_side = "buy" if side == "bid" else "sell"
        try:
            res = self.c.modify(rest["order_id"], price=want_px, qty=self.quote_qty)
        except Exception:
            # Modify failed (e.g. order already gone) -> rebuild from scratch
            self._safe_cancel(side)
            self._post(side, want_px)
            return
        o = res["order"]
        self._record_fill_from_trades(sgn_side, res.get("trades", []))
        if o["status"] in ("open", "partial"):
            self.resting[side] = {
                "order_id": o["order_id"],
                "price": o["price"],
                "qty": o["remaining"],
            }
        else:
            self.resting[side] = None

    # -------- sniping --------

    def maybe_snipe(self, fair: float, sigma: float) -> bool:
        """Take any mispriced level. Returns True if we took anything."""
        try:
            book = self.c.book(self.symbol)
        except Exception:
            return False

        edge_required = self.taker_fee + max(self.snipe_min_edge, self.snipe_buffer_sigma * sigma)
        took_any = False

        # Cheap asks: lift them
        for level in book.get("asks") or []:
            mispricing = fair - level["price"]
            if mispricing <= edge_required:
                break  # asks are price-ascending; nothing better past here
            headroom = self.position_limit - self.position
            if headroom <= 0:
                break
            qty = min(level["qty"], headroom, self.quote_qty)
            if qty <= 0:
                break
            try:
                res = self.c.buy_ioc(self.symbol, price=level["price"], qty=qty)
            except Exception:
                break
            filled = self._record_fill_from_trades("buy", res.get("trades", []))
            if filled:
                took_any = True
                print(f"  SNIPE buy  {filled} @ {level['price']}  fair={fair:.1f}  edge={mispricing:.1f}")
            else:
                break

        # Rich bids: hit them
        for level in book.get("bids") or []:
            mispricing = level["price"] - fair
            if mispricing <= edge_required:
                break
            headroom = self.position_limit + self.position
            if headroom <= 0:
                break
            qty = min(level["qty"], headroom, self.quote_qty)
            if qty <= 0:
                break
            try:
                res = self.c.sell_ioc(self.symbol, price=level["price"], qty=qty)
            except Exception:
                break
            filled = self._record_fill_from_trades("sell", res.get("trades", []))
            if filled:
                took_any = True
                print(f"  SNIPE sell {filled} @ {level['price']}  fair={fair:.1f}  edge={mispricing:.1f}")
            else:
                break

        return took_any

    # -------- top-level step --------

    def step(self, *, reconcile: bool = False) -> None:
        with self.lock:
            try:
                phase = self.c.game_state().get("phase")
            except Exception:
                return
            if phase != "running":
                # Don't try to trade outside running phase.
                if phase == "settled":
                    # Position resets at round boundary on the server side.
                    self.resting = {"bid": None, "ask": None}
                return

            if reconcile:
                self.reconcile_position()

            _, _, fair, sigma = self.desired_quotes()  # cheap; uses local posterior

            self.maybe_snipe(fair, sigma)

            # Recompute after possible snipes — position/limits may have changed.
            bid_px, ask_px, fair, sigma = self.desired_quotes()

            if bid_px is None:
                self._safe_cancel("bid")
            else:
                self._reprice("bid", bid_px)

            if ask_px is None:
                self._safe_cancel("ask")
            else:
                self._reprice("ask", ask_px)

            print(
                f"QUOTE  fv={fair:6.1f} +/-{sigma:4.1f}  pos={self.position:+4d}  "
                f"bid={bid_px}  ask={ask_px}  k={len(self.posterior.reveals)}/{self.n_total}"
            )

    # -------- event handlers --------

    def on_reveal(self, value: float) -> None:
        with self.lock:
            self.posterior.update(value)
        # Reconcile on each reveal — cheap insurance vs. local drift.
        self.step(reconcile=True)

    def on_fill_event(self, msg: dict) -> None:
        with self.lock:
            side = msg["side"]
            qty = msg["qty"]
            order_id = msg["order_id"]
            # Adjust resting bookkeeping if this fill is on one of our quotes.
            for key, expected_side in (("bid", "buy"), ("ask", "sell")):
                rest = self.resting[key]
                if rest is not None and rest["order_id"] == order_id and side == expected_side:
                    rest["qty"] -= qty
                    if rest["qty"] <= 0:
                        self.resting[key] = None
                    break
        # The WS fill is informational; we already updated position from REST trades.
        # But to be safe in case a fill arrives via WS without a corresponding REST trade
        # (e.g. our resting order got hit), reconcile here.
        self.step(reconcile=True)

    def on_phase_change(self, phase: str | None, reveals: list[float]) -> None:
        with self.lock:
            if phase == "running":
                # Start of a new round: re-init posterior with whatever reveals are showing.
                self.posterior.reset(reveals)
                self.resting = {"bid": None, "ask": None}
                self.reconcile_position()
        if phase == "running":
            self.step()

    def flatten(self) -> None:
        """Emergency: cancel all and flatten to zero via market orders."""
        with self.lock:
            try:
                self.c.cancel_all()
            except Exception:
                pass
            self.resting = {"bid": None, "ask": None}
            self.reconcile_position()
            pos = self.position
            if pos > 0:
                try:
                    res = self.c.sell_market(self.symbol, qty=pos)
                    self._record_fill_from_trades("sell", res.get("trades", []))
                except Exception as e:
                    print(f"flatten sell failed: {e}")
            elif pos < 0:
                try:
                    res = self.c.buy_market(self.symbol, qty=-pos)
                    self._record_fill_from_trades("buy", res.get("trades", []))
                except Exception as e:
                    print(f"flatten buy failed: {e}")
            print(f"FLAT  pos={self.position}")

## 4. Wire up the live feed and start

Each WS event handler is a thin shim that calls into `strat`. The strategy guards itself with a reentrant lock, so it's safe for these callbacks to fire concurrently with manual `strat.step()` calls from other cells.

In [ ]:
strat = Strategy(c, post, symbol="A")


def on_reveal(msg):
    print(f"REVEAL #{msg['index']} = {msg['value']}  running_sum={msg['running_sum']}")
    strat.on_reveal(msg["value"])


def on_fill(msg):
    print(
        f"FILL   {msg['side']:>4s} {msg['qty']} @ {msg['price']}  "
        f"liq={msg.get('liquidity')} vs {msg.get('counterparty')}"
    )
    strat.on_fill_event(msg)


def on_trade(msg):
    # Public tape — currently no-op, but useful for diagnostics
    pass


def on_book(msg):
    # We intentionally don't step on every book delta — too noisy.
    # Snipes happen on reveals/fills which is when our fair changes.
    pass


def on_game_state(msg):
    phase = msg.get("phase")
    reveals = msg.get("reveals") or []
    print(f"STATE  phase={phase}  reveals={len(reveals)}")
    strat.on_phase_change(phase, reveals)


def on_settlement(msg):
    print(f"SETTLE prices={msg.get('prices')}  pnl={msg.get('pnl')}")


def on_ack(msg):
    # Quiet by default; uncomment for debug.
    # print(f"ACK    oid={msg['order_id']} sym={msg['symbol']} status={msg['status']}")
    pass


c.on_reveal = on_reveal
c.on_fill = on_fill
c.on_trade = on_trade
c.on_book = on_book
c.on_game_state = on_game_state
c.on_settlement = on_settlement
c.on_ack = on_ack

c.start()

# If the game is already running when we start the bot, kick off an initial step.
if c.game_state().get("phase") == "running":
    strat.step(reconcile=True)

print("Bot started.")

## 5. Inspect / control

Run any of these cells while the bot is running. None of them block the WS thread.

In [ ]:
fair, sigma = strat.fair_and_sigma()
bid_px, ask_px, _, _ = strat.desired_quotes()
print(f"phase          : {c.game_state().get('phase')}")
print(f"reveals so far : {strat.posterior.reveals}")
print(f"fair value     : {fair:.2f}  +/- {sigma:.2f}")
print(f"local position : {strat.position}")
print(f"server position: {c.positions()}")
print(f"desired quotes : bid={bid_px}  ask={ask_px}")
print(f"resting        : {strat.resting}")
print(f"open orders    : {c.my_orders()}")
print(f"book           : {c.book(strat.symbol)}")

In [ ]:
# Manual nudge — force a re-evaluation right now (e.g. after tweaking knobs)
strat.step(reconcile=True)

In [ ]:
# Panic button: cancel all and market-flatten.
strat.flatten()

## 6. Design notes &amp; tuning guide

### Why this should work

- **The fair value is a true conditional expectation.** Under the model in the handout, our `predict_settle` is unbiased and asymptotically optimal. As long as nobody at the table has materially better information, we have an edge on average.
- **We charge for our own uncertainty.** `edge = max(min_edge, edge_per_sigma * sigma)` means we quote tight late (when sigma is small) and wide early (when it's large). This is the right shape: late-game fair is nearly deterministic, and a 1-tick edge is plenty of profit; early-game fair has wide uncertainty, and tight quotes would get adversely selected.
- **We snipe with a sigma-scaled buffer**, not a fixed one. If sigma is 30, we won't snipe for &lt;~10 ticks of mispricing; if sigma is 1, even 1–2 ticks of mispricing is a real edge.
- **Inventory skew is linear in position.** That's the closed-form optimum for a mean-reverting fair under quadratic inventory cost (Avellaneda-Stoikov-style).

### Knobs and what they do

All live at the top of `Strategy.__init__`. Each one is independent:

| Knob | Effect of increasing |
|---|---|
| `quote_qty` | More volume per side. Bigger fills, but bigger inventory swings and harder to manage at the position limit. |
| `min_edge` | Floor on half-spread. Raise it if you're getting picked off; lower it (toward `max(maker_fee, taker_fee) + small`) if you're not getting filled enough. |
| `edge_per_sigma` | Scales spread with uncertainty. Higher = safer but quieter. Try 0.15–0.40. |
| `skew_per_unit` | How hard we lean away from a position. Higher = mean-revert inventory faster but quote less competitively when inventoried. |
| `snipe_buffer_sigma` | Conviction needed to take. Higher = snipe rarely but profitably; lower = snipe often, risk adverse selection. |
| `snipe_min_edge` | Absolute floor on snipe profit per unit. Keep it &gt; `taker_fee`. |

### How to know if it's working

While running, watch the printed lines:

- `QUOTE fv=… +/-…` — your fair and uncertainty. Sigma should fall monotonically as reveals come in (modulo posterior revision).
- `SNIPE` — you took a mispriced level. Should be profitable on average.
- `FILL` lines from `on_fill`. `liq=maker` is good (you collected the spread); `liq=taker` was your snipe.
- `SETTLE pnl=…` at the end. Positive is the goal. Run several rounds before drawing conclusions.

### Things to consider extending

- **Cross-instrument when new symbols arrive.** When derivatives appear, derive each one's fair from the underlying's posterior. Instantiate one `Strategy` per symbol that shares the posterior on `(a, w)`.
- **Sharper posterior likelihood.** If reveals are integers (which they appear to be), the strict likelihood is `1 / (b - a + 1)` for a discrete uniform on `{a, a+1, ..., b}`, not `1 / w`. The difference is small but real; swap the `new_p = p / w` line and bracket the boundary check with `+1`.
- **React to the public tape.** `on_trade` is currently a no-op. Heavy aggressive flow in one direction is a signal about other players' fairs — could be informative if a participant is consistently more accurate than you.
- **Learn the prior across rounds.** Each round draws new `(a, w)`. Maintain a running tally of inferred `(a, w)` per round, and if the distribution looks materially different from the handout, you'll see it.